# 3. Basic Tool Calling

Learn the tool request, Python execution, and tool result loop.

In [ ]:
import os
import json
from  openai import OpenAI
import time
from pydantic import BaseModel

In [ ]:
class WeatherInput(BaseModel):
    location: str
    month: str

def get_weather(location:str, month:str) -> dict:
    print(f"Executing get_weather({location=}, {month=})")
    return {
        "location": location,
        "month": month,
        "condition": "sunny interval with possible rain",
        "temperature_c": 30,
        "rain_probability_percent": 25,
        "source": "mock test data"
    }

In [ ]:
api_key = os.getenv("AWS_BEARER_TOKEN_BEDROCK")

if not api_key:
    raise RuntimeError("API key Needed")

region = os.getenv("BEDROCK_REGION", "ap-south-1")
model_id = "google.gemma-3-4b-it"

client = OpenAI(
    api_key = api_key,
    base_url= f"https://bedrock-mantle.{region}.api.aws/v1",
)

weather_tool = {
    "type": "function",
    "function":{
        "name": "get_weather",
        "description": "Get the weather in a given location",
        "parameters":{
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "Travel destination city"
                },
                "month":{
                    "type": "string",
                    "description": "Month of travel"
                },
            },
            "required":["location", "month"],
            "additionalProperties" : False
        },
    }
}

session_usage = {
    "calls": 0,
    "input_tokens": 0,
    "output_tokens": 0,
    "total_tokens": 0,
    "latency_ms": 0,
}

In [ ]:
def call_model(
    messages: list,
    tools: list | None = None,
    tool_choice=None,
):
    start = time.perf_counter()
    request = {
        "model":model_id,
        "messages": messages,
        "temperature": 0.2,
        "max_tokens": 500,
    }

    if tools:
        request["tools"] = tools
        request["tool_choice"] = tool_choice or "auto"

    print("Tools sent:", request.get("tools"))
    print("Tool choice:", request.get("tool_choice"))
    print("Max tokens:", request.get("max_tokens"))

    response = client.chat.completions.create(**request)
    print("Finish reason:", response.choices[0].finish_reason)
    latency_ms = round((time.perf_counter() - start)* 1000)

    usage = response.usage

    session_usage["calls"] += 1
    session_usage["input_tokens"] += usage.prompt_tokens
    session_usage["output_tokens"] += usage.completion_tokens
    session_usage["total_tokens"] += usage.total_tokens
    session_usage["latency_ms"] += latency_ms

    print("\nCall usage")
    print(f"Input tokens:  {usage.prompt_tokens}")
    print(f"Output tokens: {usage.completion_tokens}")
    print(f"Total tokens:  {usage.total_tokens}")
    print(f"Latency:       {latency_ms} ms")

    print("\nSession usage")
    print(f"Calls:         {session_usage['calls']}")
    print(f"Input tokens:  {session_usage['input_tokens']}")
    print(f"Output tokens: {session_usage['output_tokens']}")
    print(f"Total tokens:  {session_usage['total_tokens']}")
    print(f"Latency:       {session_usage['latency_ms']} ms")

    return response

In [ ]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a personal decision assistant. "
            "Use the weather tool for weather-related questions. "
            "Do not invent weather information. "
            "Clearly state when information comes from mock test data."
        ),
    },
    {
        "role": "user",
        "content": (
            "I want to visit Bali in October for beaches and parties. "
            "Is the weather suitable?"
        ),
    },
]

first_response = call_model(
    messages = messages,
    tools = [weather_tool],
    tool_choice = "required"
)

assistant_message = first_response.choices[0].message
print(assistant_message)
messages.append(assistant_message)

if not assistant_message.tool_calls:
    print("\nThe model did not request a tool.")
    print(assistant_message.content)
else : 
    for tool_call in assistant_message.tool_calls:
        if tool_call.function.name != 'get_weather':
            raise ValueError(f"Tool Call is not allowed: {tool_call.function.name}")

        weather_input = WeatherInput.model_validate_json(
            tool_call.function.arguments
        )
        weather_result = get_weather(
            location = weather_input.location,
            month = weather_input.month,
        )

        print("\nTool requested")
        print(tool_call.function.name)
        print(weather_input.model_dump_json(indent=2))

        print("\nTool result")
        print(json.dumps(weather_result, indent=2))

        messages.append(
            {
                "role": "user",
                "content": (
                    "The get_weather tool returned this result:\n"
                    f"{json.dumps(weather_result)}\n"
                    "Use only this tool result to answer the original question."
                ),
            }
        )

        final_response = call_model(messages = messages)
        final_answer = final_response.choices[0].message.content

        print("\nFinal answer")
        print(final_answer)

